# LSTM DQN Training - IMC Prosperity 4

**What this notebook does:**
1. Installs dependencies (stable-baselines3, gymnasium)
2. Uploads/mounts your trading data
3. Computes 19 trading indicators from order book data
4. Trains an LSTM-based DQN agent with a custom PyTorch training loop
5. Evaluates on held-out data and exports weights for submission

**Run this on:** Google Colab (free GPU) or Kaggle Notebooks

---

## 1. Setup & Install Dependencies

In [ ]:
!pip install stable-baselines3 gymnasium pandas numpy matplotlib --quiet
print("Dependencies installed!")

## 2. Setup Data (Google Colab)

In [ ]:
# === Google Colab Setup (DEFAULT) ===
# Clone your repo which already has the data and code
!git clone https://github.com/YOUR_USERNAME/EnM-Making-Waves---Prosperity-4.git /content/repo
DATA_DIR = "/content/repo/data"
PROJECT_ROOT = "/content/repo"

# === Alternative: Google Drive mount ===
# from google.colab import drive
# drive.mount('/content/drive')
# DATA_DIR = "/content/drive/MyDrive/EnM-Making-Waves---Prosperity-4/data"
# PROJECT_ROOT = "/content/drive/MyDrive/EnM-Making-Waves---Prosperity-4"

# === Alternative: Kaggle ===
# DATA_DIR = "/kaggle/input/prosperity4/data"
# PROJECT_ROOT = "/kaggle/input/prosperity4"

import sys
sys.path.insert(0, PROJECT_ROOT)
print(f"Data dir: {DATA_DIR}")
print(f"Project root: {PROJECT_ROOT}")

## 3. Load Data & Check GPU

In [ ]:
import torch
import numpy as np
import pandas as pd
import os

# Check GPU availability
if torch.cuda.is_available():
    print(f"GPU available: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("No GPU detected - training on CPU (still fast for this model)")

# Load data
from RLM.shared.data_loader import load_prices, load_trades

prices_df = load_prices(data_dir=DATA_DIR)
trades_df = load_trades(data_dir=DATA_DIR)

print(f"\nPrices: {len(prices_df)} rows")
print(f"Trades: {len(trades_df)} rows")
print(f"Products: {prices_df['product'].unique()}")
print(f"Days: {sorted(prices_df['day'].unique())}")

## 4. Compute Feature Normalization

Before training, we need to compute the mean and standard deviation of each feature across the training data. This ensures all 19 features are on the same scale.

In [ ]:
from RLM.shared.config import PRODUCTS, TRAIN_CONFIG, DQN_CONFIG, NETWORK_CONFIG, ENV_CONFIG
from RLM.shared.features import FeatureComputer, compute_features_from_row, fit_normalizer
from RLM.shared.env import TradingEnv

# Compute normalization parameters from training data
def compute_normalization_params(prices_df, trades_df, products, days):
    all_features = {p: [] for p in products}
    for day in days:
        for product in products:
            fc = FeatureComputer(product)
            day_prices = prices_df[(prices_df["day"] == day) & (prices_df["product"] == product)]
            day_prices = day_prices.sort_values("timestamp").reset_index(drop=True)
            day_trades = trades_df[trades_df["symbol"] == product].sort_values("timestamp")
            for _, row in day_prices.iterrows():
                ts = row["timestamp"]
                ts_trades = day_trades[day_trades["timestamp"] == ts]
                trades = list(zip(ts_trades["price"], ts_trades["quantity"])) if len(ts_trades) > 0 else None
                features = compute_features_from_row(row, fc, position=0, trades=trades)
                all_features[product].append(features)
    combined = np.vstack([np.array(v) for v in all_features.values()])
    return fit_normalizer(combined)

feat_means, feat_stds = compute_normalization_params(
    prices_df, trades_df, PRODUCTS, TRAIN_CONFIG["train_days"]
)

print("Feature normalization computed!")
print(f"Means (first 5): {feat_means[:5]}")
print(f"Stds  (first 5): {feat_stds[:5]}")

## 5. Train LSTM DQN (Custom Training Loop)

Uses a custom PyTorch training loop (not SB3) since SB3's DQN does not natively support recurrent networks. The LSTM maintains hidden state across timesteps within an episode.

**Adjust `TOTAL_TIMESTEPS`, `HIDDEN_SIZE`, and `SEQ_LEN` below.**

In [ ]:
# ============================================================
# TRAINING CONFIG - Adjust these!
# ============================================================
TOTAL_TIMESTEPS = 100_000   # Increase for better results (200k, 500k)
HIDDEN_SIZE = 64            # LSTM hidden dimension
SEQ_LEN = 10                # Sequence length for replay buffer
SEED = 42
LEARNING_RATE = 1e-3        # Set to None to use default from config
# ============================================================

import torch
import torch.nn as nn
import torch.optim as optim
import random
from collections import deque
from RLM.lstm_dqn.train import LSTMQNetwork, SequenceReplayBuffer
from RLM.shared.config import NUM_FEATURES, NUM_ACTIONS

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# Create environments
train_env = TradingEnv(
    prices_df=prices_df, trades_df=trades_df,
    products=PRODUCTS, day=TRAIN_CONFIG["train_days"][0],
    augment=True, seed=SEED,
)
eval_env = TradingEnv(
    prices_df=prices_df, trades_df=trades_df,
    products=PRODUCTS, day=TRAIN_CONFIG["eval_days"][0],
    augment=False, seed=SEED + 1,
)

# Set normalization params
for product in PRODUCTS:
    train_env.feature_computers[product].feature_means = feat_means
    train_env.feature_computers[product].feature_stds = feat_stds
    eval_env.feature_computers[product].feature_means = feat_means
    eval_env.feature_computers[product].feature_stds = feat_stds

# Output directory
MODEL_DIR = os.path.join(PROJECT_ROOT, "RLM", "lstm_dqn", "policy_weights")
os.makedirs(MODEL_DIR, exist_ok=True)

# Dimensions
obs_dim = NUM_FEATURES * len(PRODUCTS)
action_dim = NUM_ACTIONS ** len(PRODUCTS)

# Create networks
q_net = LSTMQNetwork(obs_dim, action_dim, HIDDEN_SIZE).to(device)
target_net = LSTMQNetwork(obs_dim, action_dim, HIDDEN_SIZE).to(device)
target_net.load_state_dict(q_net.state_dict())

optimizer = optim.Adam(q_net.parameters(), lr=LEARNING_RATE or DQN_CONFIG["learning_rate"])
replay_buffer = SequenceReplayBuffer(DQN_CONFIG["buffer_size"], seq_len=SEQ_LEN)

# Training loop
total_steps = 0
best_eval_pnl = -float("inf")
epsilon = DQN_CONFIG["exploration_initial_eps"]
eps_decay = (DQN_CONFIG["exploration_initial_eps"] - DQN_CONFIG["exploration_final_eps"]) / \
            (TOTAL_TIMESTEPS * DQN_CONFIG["exploration_fraction"])

print(f"\n  Hidden size: {HIDDEN_SIZE}")
print(f"  Sequence length: {SEQ_LEN}")
print(f"  Total timesteps: {TOTAL_TIMESTEPS}")
print("\nTraining...")

episode = 0
while total_steps < TOTAL_TIMESTEPS:
    obs, info = train_env.reset()
    hidden = q_net.init_hidden(1, device)
    done = False
    ep_reward = 0

    while not done and total_steps < TOTAL_TIMESTEPS:
        # Epsilon-greedy action selection
        if random.random() < epsilon:
            action = train_env.action_space.sample()
        else:
            with torch.no_grad():
                obs_t = torch.FloatTensor(obs).unsqueeze(0).to(device)
                q_values, hidden = q_net(obs_t, hidden)
                action = q_values.argmax(dim=-1).item()

        next_obs, reward, terminated, truncated, info = train_env.step(action)
        done = terminated or truncated
        replay_buffer.add(obs, action, reward, next_obs, done)

        obs = next_obs
        ep_reward += reward
        total_steps += 1
        epsilon = max(DQN_CONFIG["exploration_final_eps"], epsilon - eps_decay)

        # Training step
        if len(replay_buffer) >= DQN_CONFIG["batch_size"] and total_steps % DQN_CONFIG["train_freq"] == 0:
            sequences = replay_buffer.sample(DQN_CONFIG["batch_size"])

            loss_total = 0
            for seq in sequences:
                states = torch.FloatTensor([s[0] for s in seq]).unsqueeze(0).to(device)
                actions = torch.LongTensor([s[1] for s in seq]).to(device)
                rewards = torch.FloatTensor([s[2] for s in seq]).to(device)
                next_states = torch.FloatTensor([s[3] for s in seq]).unsqueeze(0).to(device)
                dones = torch.FloatTensor([float(s[4]) for s in seq]).to(device)

                h = q_net.init_hidden(1, device)
                q_values, _ = q_net(states, h)
                q_values = q_values.squeeze(0)

                # Only use last transition for loss
                q_value = q_values[actions[-1]]

                with torch.no_grad():
                    h_t = target_net.init_hidden(1, device)
                    next_q, _ = target_net(next_states, h_t)
                    next_q = next_q.squeeze(0)
                    target = rewards[-1] + DQN_CONFIG["gamma"] * next_q.max() * (1 - dones[-1])

                loss_total += (q_value - target) ** 2

            loss = loss_total / len(sequences)
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(q_net.parameters(), 10.0)
            optimizer.step()

        # Update target network
        if total_steps % DQN_CONFIG["target_update_interval"] == 0:
            target_net.load_state_dict(q_net.state_dict())

    episode += 1
    if episode % 5 == 0:
        print(f"  Episode {episode}: steps={total_steps}, reward={ep_reward:.2f}, eps={epsilon:.3f}")

    # Periodic evaluation
    if episode % 10 == 0:
        eval_obs, _ = eval_env.reset()
        eval_hidden = q_net.init_hidden(1, device)
        eval_done = False
        eval_reward = 0
        while not eval_done:
            with torch.no_grad():
                obs_t = torch.FloatTensor(eval_obs).unsqueeze(0).to(device)
                eval_q, eval_hidden = q_net(obs_t, eval_hidden)
                eval_action = eval_q.argmax(dim=-1).item()
            eval_obs, r, term, trunc, eval_info = eval_env.step(eval_action)
            eval_reward += r
            eval_done = term or trunc
        eval_pnl = eval_info.get("pnl", eval_reward)
        print(f"  [EVAL] PnL={eval_pnl:.2f}")
        if eval_pnl > best_eval_pnl:
            best_eval_pnl = eval_pnl
            torch.save(q_net.state_dict(), os.path.join(MODEL_DIR, "best_model.pt"))
            print(f"  [EVAL] New best! Saved.")

# Save final model
torch.save(q_net.state_dict(), os.path.join(MODEL_DIR, "final_model.pt"))

# Save metadata
np.savez(os.path.join(MODEL_DIR, "norm_params.npz"),
         feature_means=feat_means, feature_stds=feat_stds)
np.savez(os.path.join(MODEL_DIR, "model_config.npz"),
         hidden_size=np.array(HIDDEN_SIZE),
         obs_dim=np.array(obs_dim),
         action_dim=np.array(action_dim))

print(f"\nTraining complete! Best eval PnL: {best_eval_pnl:.2f}")
print(f"Models saved in: {MODEL_DIR}")

## 6. Evaluate on Held-Out Day

Run the trained LSTM agent on the eval day (which it never saw during training) and check PnL.

In [ ]:
# Evaluate on held-out data
import matplotlib.pyplot as plt

# Load best model
best_path = os.path.join(MODEL_DIR, "best_model.pt")
if os.path.exists(best_path):
    q_net.load_state_dict(torch.load(best_path, map_location=device))
    print("Loaded best model for evaluation")

n_eval_episodes = 10
episode_pnls = []
episode_rewards = []

for ep in range(n_eval_episodes):
    obs, info = eval_env.reset()
    hidden = q_net.init_hidden(1, device)
    total_reward = 0
    done = False

    while not done:
        with torch.no_grad():
            obs_t = torch.FloatTensor(obs).unsqueeze(0).to(device)
            q_values, hidden = q_net(obs_t, hidden)
            action = q_values.argmax(dim=-1).item()

        obs, reward, terminated, truncated, info = eval_env.step(action)
        total_reward += reward
        done = terminated or truncated

    episode_pnls.append(info.get("pnl", total_reward))
    episode_rewards.append(total_reward)
    print(f"Episode {ep+1}: PnL={episode_pnls[-1]:.2f}, Positions={info.get('positions', {})}")

pnls = np.array(episode_pnls)
print(f"\n{'='*40}")
print(f"Mean PnL:  {pnls.mean():.2f}")
print(f"Std PnL:   {pnls.std():.2f}")
print(f"Min PnL:   {pnls.min():.2f}")
print(f"Max PnL:   {pnls.max():.2f}")
if pnls.std() > 0:
    print(f"Sharpe:    {pnls.mean() / pnls.std():.2f}")

## 7. Export Weights to Numpy

Extract LSTM gate weights (lstm_W_ii, lstm_W_if, etc.) and MLP head weights (head_W0, head_B0, etc.) into a `.npz` file for the NumpyLSTM inference class.

In [ ]:
# Export LSTM DQN weights to numpy .npz
weights = {}
weights["lstm_hidden_size"] = np.array(HIDDEN_SIZE)

# Extract LSTM weights
# PyTorch LSTM packs all 4 gates into single weight matrices
# weight_ih_l0: (4*hidden, input) -- gates: i, f, g, o
# weight_hh_l0: (4*hidden, hidden)
# bias_ih_l0: (4*hidden,)
# bias_hh_l0: (4*hidden,)
lstm = q_net.lstm

W_ih = lstm.weight_ih_l0.detach().cpu().numpy()
W_hh = lstm.weight_hh_l0.detach().cpu().numpy()
b_ih = lstm.bias_ih_l0.detach().cpu().numpy()
b_hh = lstm.bias_hh_l0.detach().cpu().numpy()

H = HIDDEN_SIZE
gate_names = ["i", "f", "g", "o"]  # input, forget, cell, output

for idx, gate in enumerate(gate_names):
    weights[f"lstm_W_i{gate}"] = W_ih[idx * H:(idx + 1) * H]
    weights[f"lstm_W_h{gate}"] = W_hh[idx * H:(idx + 1) * H]
    weights[f"lstm_b_{gate}"] = b_ih[idx * H:(idx + 1) * H] + b_hh[idx * H:(idx + 1) * H]
    print(f"  Gate '{gate}': W_i{gate} {weights[f'lstm_W_i{gate}'].shape}, "
          f"W_h{gate} {weights[f'lstm_W_h{gate}'].shape}, "
          f"b_{gate} {weights[f'lstm_b_{gate}'].shape}")

# Extract MLP head weights
head_idx = 0
for name, param in q_net.head.named_parameters():
    tensor = param.detach().cpu().numpy()
    print(f"  head.{name}: {tensor.shape}")
    if "weight" in name:
        weights[f"head_W{head_idx}"] = tensor
    elif "bias" in name:
        weights[f"head_B{head_idx}"] = tensor
        head_idx += 1

# Include normalization params
weights["feature_means"] = feat_means
weights["feature_stds"] = feat_stds

# Save
export_path = os.path.join(MODEL_DIR, "exported_weights.npz")
np.savez(export_path, **weights)

total_params = sum(v.size for k, v in weights.items() if "feature" not in k and "hidden_size" not in k)
file_size = os.path.getsize(export_path)
print(f"\nExported {len(weights)} arrays, {total_params:,} parameters")
print(f"File size: {file_size / 1024:.1f} KB")
print(f"Saved to: {export_path}")

# Verify with numpy forward pass
from RLM.shared.numpy_policy import NumpyLSTM
numpy_model = NumpyLSTM(weights_path=export_path)
test_obs = np.random.randn(obs_dim).astype(np.float32)
action, q_vals = numpy_model.predict(test_obs, normalize=False)
print(f"\nVerification pass: action={action}, Q-values={q_vals[:3]}...")

## 8. Download Weights

Run this cell to download the exported weights to your local machine. Then use `build_submission.py` locally to create the submission file.

In [ ]:
# Download weights (Google Colab only)
from google.colab import files
files.download(export_path)
print("Download started! Check your browser downloads.")